# 참가자용 Starter Baseline

이 노트북은 각 조가 직접 구축한 **FLAME 기반 YOLO 데이터셋**(`datasets/`와 `data.yaml`)으로 객체 탐지 모델을 학습하기 위한 시작점입니다. 예시 모델은 YOLOv8m이지만 필수 모델은 아니며, 조별 실험 목적에 따라 다른 모델을 사용해도 됩니다.

기존 EXP-001의 재현 기록과 실행 절차는 `colab/EXP-001-baseline.ipynb`에 별도로 보존되어 있습니다.

## 1. Repository 준비

Colab의 `/content/AnomalyDetection`에 저장소가 없으면 자동으로 clone하고, 이후 작업 위치를 저장소 루트로 변경합니다.

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess

REPO_DIR = Path("/content/AnomalyDetection")
REPO_URL = "https://github.com/adeomise/AnomalyDetection.git"

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
elif not (REPO_DIR / ".git").exists():
    raise RuntimeError(f"{REPO_DIR} 폴더는 있지만 Git 저장소가 아닙니다. 폴더를 확인해 주세요.")

os.chdir(REPO_DIR)
print("현재 작업 폴더:", Path.cwd())

## 2. 환경 설정과 상태 확인

EXP-001용 `scripts/colab/setup_exp001.sh`는 `/content/baseline-env`라는 별도 환경과 EXP-001 전용 의존성 및 검증 절차를 구성합니다. 참가자 데이터에는 그 전용 환경을 실행하지 않고, 현재 Colab 커널에 Ultralytics를 설치합니다. 이렇게 하면 별도 환경의 Python과 노트북 커널이 섞이는 문제를 피할 수 있으며 Roboflow도 필요하지 않습니다.

설치 후 Python, PyTorch, CUDA, Ultralytics 및 GPU 상태를 확인합니다. Colab 메뉴에서 GPU 런타임을 선택하지 않았다면 CUDA 사용 가능 여부가 `False`로 표시될 수 있습니다.

In [ ]:
%pip install -q ultralytics

In [ ]:
import platform
import subprocess
import torch
import ultralytics

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("PyTorch CUDA 빌드:", torch.version.cuda)
print("Ultralytics:", ultralytics.__version__)
print("CUDA 사용 가능:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU를 찾지 못했습니다. Colab 런타임 유형을 확인해 주세요.")

if shutil.which("nvidia-smi"):
    gpu_status = subprocess.run(
        ["nvidia-smi"], capture_output=True, text=True, check=False
    )
    print(gpu_status.stdout if gpu_status.returncode == 0 else "nvidia-smi 실행에 실패했습니다.")
else:
    print("nvidia-smi를 사용할 수 없습니다.")

## 3. 데이터셋 설정

아래 `DATA_YAML`을 각 조의 `data.yaml` 실제 경로로 수정하세요. 기본 예시는 `/content/data.yaml`입니다. Google Drive를 마운트했다면 `/content/drive/MyDrive/.../data.yaml` 같은 경로도 사용할 수 있습니다.

`data.yaml`의 `train`, `val`, 선택 항목인 `test` 경로도 실제 이미지 또는 이미지 목록 위치와 일치해야 합니다. 상대 경로를 쓴 경우 `path` 설정과 YAML 파일 위치를 함께 확인하세요.

In [ ]:
from pathlib import Path

# 각 조의 data.yaml 경로로 수정하세요.
DATA_YAML = Path("/content/data.yaml")
print("사용할 data.yaml:", DATA_YAML)

## 4. 데이터셋 연결 확인

YAML 파일과 `train`/`val` 기본 항목을 확인하고, 로컬 경로가 실제로 연결되는지 점검합니다. 데이터셋 크기나 특정 이미지 개수는 강제하지 않습니다. URL처럼 원격 위치를 쓴 항목은 로컬 존재 여부 검사 대신 설정값만 표시합니다.

In [ ]:
import glob
import yaml

DATA_YAML = DATA_YAML.expanduser().resolve()
if not DATA_YAML.is_file():
    raise FileNotFoundError(
        f"data.yaml을 찾을 수 없습니다: {DATA_YAML}\n"
        "DATA_YAML을 실제 파일 경로로 수정한 뒤 다시 실행해 주세요."
    )

with DATA_YAML.open(encoding="utf-8") as file:
    data_config = yaml.safe_load(file) or {}

missing_keys = [key for key in ("train", "val") if not data_config.get(key)]
if missing_keys:
    raise ValueError(f"data.yaml에 필수 항목이 없습니다: {missing_keys}")

yaml_dir = DATA_YAML.parent
configured_root = Path(str(data_config.get("path", yaml_dir))).expanduser()
dataset_root = configured_root if configured_root.is_absolute() else yaml_dir / configured_root
dataset_root = dataset_root.resolve()
print("데이터셋 기준 폴더:", dataset_root)

def check_split(split_name, configured_value):
    # 여러 경로를 목록으로 지정한 YAML도 처리합니다.
    values = configured_value if isinstance(configured_value, list) else [configured_value]
    for value in values:
        value = str(value)
        if value.startswith(("http://", "https://")):
            print(f"{split_name}: 원격 경로 - {value}")
            continue
        candidate = Path(value).expanduser()
        candidate = candidate if candidate.is_absolute() else dataset_root / candidate
        matched = glob.glob(str(candidate)) if any(char in str(candidate) for char in "*?[]") else []
        exists = candidate.exists() or bool(matched)
        print(f"{split_name}: {'확인됨' if exists else '찾을 수 없음'} - {candidate}")
        if not exists:
            raise FileNotFoundError(
                f"{split_name} 경로를 찾을 수 없습니다. data.yaml과 실제 데이터 위치를 확인하세요: {candidate}"
            )

for split in ("train", "val", "test"):
    if data_config.get(split):
        check_split(split, data_config[split])

print("data.yaml 기본 연결 확인을 마쳤습니다.")

## 5. 학습 설정

실험 전에 아래 값만 쉽게 수정할 수 있습니다.

- `MODEL_NAME`: 시작할 사전 학습 가중치 또는 모델 설정입니다. `yolov8m.pt`는 참고 baseline입니다.
- `EPOCHS`: 전체 학습 데이터를 반복할 횟수입니다.
- `IMG_SIZE`: 학습 입력 이미지 크기입니다. 큰 값은 더 많은 GPU 메모리가 필요할 수 있습니다.
- `BATCH_SIZE`: 한 번에 처리할 이미지 수입니다. GPU 메모리 부족 시 줄이세요.

In [ ]:
MODEL_NAME = "yolov8m.pt"
EPOCHS = 50
IMG_SIZE = 640
BATCH_SIZE = 16

print({
    "model": MODEL_NAME,
    "epochs": EPOCHS,
    "image_size": IMG_SIZE,
    "batch_size": BATCH_SIZE,
})

## 6. 학습

참가자의 `data.yaml`을 Ultralytics YOLO에 직접 전달합니다. 결과는 `/content/fire-training/baseline`에 저장됩니다. 이 셀을 실행하면 실제 학습이 시작되므로 설정과 GPU 상태를 먼저 확인하세요.

In [ ]:
from ultralytics import YOLO

model = YOLO(MODEL_NAME)
train_results = model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    project="/content/fire-training",
    name="baseline",
    exist_ok=True,
)

## 7. 결과 확인

`best.pt`는 검증 지표가 가장 좋았던 epoch의 가중치이고, `last.pt`는 마지막 epoch가 끝난 시점의 가중치입니다. 아래 셀은 실제 학습 결과 폴더에서 두 경로를 확인합니다.

In [ ]:
from pathlib import Path

RUN_DIR = Path(model.trainer.save_dir)
BEST_WEIGHTS = RUN_DIR / "weights" / "best.pt"
LAST_WEIGHTS = RUN_DIR / "weights" / "last.pt"

print("학습 결과 폴더:", RUN_DIR)
print("best.pt (검증 성능이 가장 좋았던 가중치):", BEST_WEIGHTS)
print("last.pt (마지막 epoch의 가중치):", LAST_WEIGHTS)
print("best.pt 존재 여부:", BEST_WEIGHTS.is_file())
print("last.pt 존재 여부:", LAST_WEIGHTS.is_file())

## 8. 선택 추론 예제

추론할 이미지 또는 영상의 실제 경로를 `INFERENCE_SOURCE`에 입력하세요. 빈 문자열이거나 파일이 없으면 오류 대신 안내 메시지를 출력합니다. 기본 제공 샘플 파일이 있다고 가정하지 않습니다.

In [ ]:
from pathlib import Path
from ultralytics import YOLO

# 추론할 이미지 또는 영상의 실제 경로를 입력하세요.
INFERENCE_SOURCE = ""

source_path = Path(INFERENCE_SOURCE).expanduser() if INFERENCE_SOURCE.strip() else None
if source_path is None:
    print("추론을 건너뜁니다. INFERENCE_SOURCE에 이미지 또는 영상 경로를 입력하세요.")
elif not source_path.is_file():
    print(f"파일을 찾을 수 없습니다: {source_path}")
    print("Google Drive 마운트 여부와 입력 경로를 확인해 주세요.")
elif not BEST_WEIGHTS.is_file():
    print(f"학습 가중치를 찾을 수 없습니다: {BEST_WEIGHTS}")
    print("먼저 학습 셀을 완료하거나 사용할 가중치 경로를 지정해 주세요.")
else:
    inference_model = YOLO(str(BEST_WEIGHTS))
    prediction_results = inference_model.predict(
        source=str(source_path),
        save=True,
        project="/content/fire-training",
        name="baseline-predict",
        exist_ok=True,
    )
    print("추론 결과 저장 폴더:", prediction_results[0].save_dir)

## 9. 다음 실험 안내

YOLOv8m은 비교를 위한 선택 가능한 baseline일 뿐 사용을 강제하지 않습니다. augmentation, 입력 이미지 크기, epoch, 추가 데이터 및 모델을 자유롭게 변경해 실험할 수 있습니다. Faster R-CNN, DETR 등 다른 Object Detection 모델도 사용할 수 있습니다. 다만 다른 모델을 선택하면 해당 모델의 학습 코드와 realtime 추론 코드는 각 조가 직접 구성해야 합니다.